# Optical & Spectral Sandbox: 1 cm Warp & Weft Thread Counting

**Objective:** Mathematically verify and implement the dual-engine extraction of fabric thread density (Warp Ends/cm and Weft Picks/cm) using 2D Fast Fourier Transform (2D FFT) and 1D Directional Spatial Projection Profiling.

**The mechanistic justification for this step is:** Provide an ultra-lightweight, zero-download, offline computer vision engine runnable on standard laptop CPUs without requiring heavy deep-learning model downloads (>6 GB), while maintaining sub-pixel thread density precision.

In [1]:
# The mechanistic justification for this step is: configure the global Kaggle pathing invariant with local fallback routing and initialize PyTorch with native NumPy fallback capabilities.
import os
import json
import numpy as np
from PIL import Image

# 1. Environment and Kaggle Pathing Invariant
INPUT_DIR = "/kaggle/input/fabric-data" if os.path.exists("/kaggle/input/fabric-data") else "data"
WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "results"
os.makedirs(WORKING_DIR, exist_ok=True)
os.makedirs(os.path.join(INPUT_DIR, "synthetic"), exist_ok=True)
os.makedirs(os.path.join(INPUT_DIR, "samples"), exist_ok=True)

# 2. Framework Detection (PyTorch preferred, NumPy fallback)
try:
    import torch
    TORCH_AVAILABLE = True
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
except ImportError:
    TORCH_AVAILABLE = False
    DEVICE = "cpu"

print(f"Framework: {'PyTorch (' + str(DEVICE) + ')' if TORCH_AVAILABLE else 'NumPy (CPU)'}")
print(f"Input Directory: {INPUT_DIR}")
print(f"Working Directory: {WORKING_DIR}")

Framework: NumPy (CPU)
Input Directory: data
Working Directory: results


## 1. Synthetic Fabric Lattice Generation with Known Ground Truth

To enable rigorous mathematical verification on any hardware without requiring external network downloads, we simulate a calibrated 1.0 cm x 1.0 cm woven fabric patch with known ground truth ($N_{\text{warp}} = 32\text{ ends/cm}$, $N_{\text{weft}} = 28\text{ picks/cm}$, Scale $= 400\text{ pixels/cm}$).

In [2]:
# The mechanistic justification for this step is: simulate physical woven fabric reflectance with sinusoidal yarn crowns, interlacing phase modulation, and Gaussian fibril hairiness noise.
pixels_per_cm = 400
img_size = 400  # 1 cm x 1 cm ROI at 400 px/cm
true_warp_count = 32  # 32 ends/cm
true_weft_count = 28  # 28 picks/cm

x = np.linspace(0, 1, img_size, endpoint=False)
y = np.linspace(0, 1, img_size, endpoint=False)
X, Y = np.meshgrid(x, y)

# Sinusoidal surface crowns for warp and weft yarns
warp_signal = 0.5 * (1.0 + np.cos(2.0 * np.pi * true_warp_count * X))
weft_signal = 0.5 * (1.0 + np.cos(2.0 * np.pi * true_weft_count * Y))

# 1x1 plain weave interlacing phase modulation
interlace = np.sin(2.0 * np.pi * true_warp_count * X) * np.sin(2.0 * np.pi * true_weft_count * Y)
fabric_texture = 0.45 * warp_signal + 0.45 * weft_signal + 0.1 * interlace

# Add Gaussian noise (simulating surface hairiness and fibril scattering)
np.random.seed(42)
noise = np.random.normal(0, 0.05, (img_size, img_size))
fabric_texture = np.clip(fabric_texture + noise, 0.0, 1.0)
fabric_uint8 = (fabric_texture * 255.0).astype(np.uint8)

# Save synthetic benchmark sample
synthetic_sample_path = os.path.join(INPUT_DIR, "synthetic", "sample_plain_32x28.png")
Image.fromarray(fabric_uint8).save(synthetic_sample_path)

print(f"Generated sample shape: {fabric_texture.shape}")
print(f"Calibrated Scale: {pixels_per_cm} pixels/cm")
print(f"Ground Truth: Warp = {true_warp_count} ends/cm | Weft = {true_weft_count} picks/cm")
print(f"Saved to: {synthetic_sample_path}")

Generated sample shape: (400, 400)
Calibrated Scale: 400 pixels/cm
Ground Truth: Warp = 32 ends/cm | Weft = 28 picks/cm
Saved to: data/synthetic/sample_plain_32x28.png


## 2. Engine 1: 2D Fast Fourier Transform (2D FFT) Spectral Analysis

In the 2D frequency spectrum, a periodic yarn lattice produces distinct orthogonal spectral impulses at spatial frequencies $(u_0, v_0)$:

$$\text{Ends/cm} = u_0 \cdot \frac{S}{N_x}, \quad \text{Picks/cm} = v_0 \cdot \frac{S}{N_y}$$

where $S$ is the scale factor (pixels/cm) and $N_x, N_y$ are the ROI dimensions.

In [3]:
# The mechanistic justification for this step is: extract fundamental harmonic frequencies of orthogonal yarn lattices by eliminating DC bias and computing the centered 2D Fourier power spectrum.
# Remove DC bias by subtracting image mean
f_input = fabric_texture - np.mean(fabric_texture)

if TORCH_AVAILABLE:
    t_input = torch.tensor(f_input, dtype=torch.float32, device=DEVICE)
    fft2 = torch.fft.fft2(t_input)
    fft_shift = torch.fft.fftshift(fft2)
    magnitude = torch.abs(fft_shift).cpu().numpy()
else:
    fft2 = np.fft.fft2(f_input)
    fft_shift = np.fft.fftshift(fft2)
    magnitude = np.abs(fft_shift)

rows, cols = magnitude.shape
crow, ccol = rows // 2, cols // 2

# Search range: filter out low-frequency illumination gradients (< 5 cycles/cm) up to Nyquist limit
min_freq = 5
max_freq = int(pixels_per_cm / 2)

# Horizontal spectrum slice (Warp thread frequency along x-axis)
horiz_spectrum = np.mean(magnitude[crow - 2 : crow + 3, ccol + min_freq : ccol + max_freq], axis=0)
pred_warp_fft = int(np.argmax(horiz_spectrum) + min_freq)

# Vertical spectrum slice (Weft thread frequency along y-axis)
vert_spectrum = np.mean(magnitude[crow + min_freq : crow + max_freq, ccol - 2 : ccol + 3], axis=1)
pred_weft_fft = int(np.argmax(vert_spectrum) + min_freq)

print(f"2D FFT Magnitude Spectrum Shape: {magnitude.shape}")
print(f"Predicted Warp Threads (FFT): {pred_warp_fft} ends/cm (Error: {abs(pred_warp_fft - true_warp_count)})")
print(f"Predicted Weft Threads (FFT): {pred_weft_fft} picks/cm (Error: {abs(pred_weft_fft - true_weft_count)})")

2D FFT Magnitude Spectrum Shape: (400, 400)
Predicted Warp Threads (FFT): 32 ends/cm (Error: 0)
Predicted Weft Threads (FFT): 28 picks/cm (Error: 0)


## 3. Engine 2: 1D Spatial Projection Profiling & Peak Prominence

By integrating pixel intensities along the orthogonal axes:

$$P_{\text{warp}}(x) = \frac{1}{H} \sum_{y=1}^{H} I(x, y), \quad P_{\text{weft}}(y) = \frac{1}{W} \sum_{x=1}^{W} I(x, y)$$

peaks in $P_{\text{warp}}(x)$ correspond to individual vertical warp yarn crowns, while peaks in $P_{\text{weft}}(y)$ correspond to horizontal weft yarn crowns.

In [4]:
# The mechanistic justification for this step is: integrate 2D image intensities into 1D marginal projection vectors and extract local crown maxima with adaptive minimum inter-yarn spacing.
warp_profile = np.mean(fabric_texture, axis=0)
weft_profile = np.mean(fabric_texture, axis=1)

def count_1d_peaks(signal, min_distance=5):
    peaks = []
    for i in range(1, len(signal) - 1):
        if signal[i] > signal[i - 1] and signal[i] > signal[i + 1]:
            if len(peaks) == 0 or (i - peaks[-1]) >= min_distance:
                peaks.append(i)
            elif signal[i] > signal[peaks[-1]]:
                peaks[-1] = i
    return np.array(peaks)

# Adaptive minimum distance based on FFT frequency prior
min_warp_dist = max(3, int(pixels_per_cm / (pred_warp_fft * 1.5)))
min_weft_dist = max(3, int(pixels_per_cm / (pred_weft_fft * 1.5)))

warp_peaks = count_1d_peaks(warp_profile, min_distance=min_warp_dist)
weft_peaks = count_1d_peaks(weft_profile, min_distance=min_weft_dist)

pred_warp_proj = int(len(warp_peaks))
pred_weft_proj = int(len(weft_peaks))

# Calculate mean thread spacing in microns
mean_warp_spacing_px = np.mean(np.diff(warp_peaks))
mean_weft_spacing_px = np.mean(np.diff(weft_peaks))
mean_warp_spacing_um = (mean_warp_spacing_px / pixels_per_cm) * 10000.0
mean_weft_spacing_um = (mean_weft_spacing_px / pixels_per_cm) * 10000.0

print(f"Predicted Warp Threads (Projection): {pred_warp_proj} ends/cm (Spacing: {mean_warp_spacing_um:.1f} um)")
print(f"Predicted Weft Threads (Projection): {pred_weft_proj} picks/cm (Spacing: {mean_weft_spacing_um:.1f} um)")

Predicted Warp Threads (Projection): 31 ends/cm (Spacing: 313.3 um)
Predicted Weft Threads (Projection): 27 picks/cm (Spacing: 357.7 um)


## 4. Consensus Engine & Output Persistence

The consensus engine compares results across both engines. If $|N_{\text{FFT}} - N_{\text{proj}}| \le 1$, the measurement is validated. Raw outputs are irrevocably written to `results/output.json` and `results/output.txt`.

In [5]:
# The mechanistic justification for this step is: validate algorithmic consistency across spectral and spatial domains and persist raw results to disk per project memory invariants.
delta_warp = abs(pred_warp_fft - pred_warp_proj)
delta_weft = abs(pred_weft_fft - pred_weft_proj)

final_warp_count = pred_warp_fft if delta_warp <= 1 else int(round((pred_warp_fft + pred_warp_proj) / 2.0))
final_weft_count = pred_weft_fft if delta_weft <= 1 else int(round((pred_weft_fft + pred_weft_proj) / 2.0))

results_payload = {
    "calibration": {
        "pixels_per_cm": pixels_per_cm,
        "pixel_size_um": 10000.0 / pixels_per_cm,
        "roi_dimensions": [img_size, img_size]
    },
    "ground_truth": {
        "warp_ends_per_cm": true_warp_count,
        "weft_picks_per_cm": true_weft_count
    },
    "spectral_fft_engine": {
        "warp_ends_per_cm": pred_warp_fft,
        "weft_picks_per_cm": pred_weft_fft,
        "error_warp": abs(pred_warp_fft - true_warp_count),
        "error_weft": abs(pred_weft_fft - true_weft_count)
    },
    "spatial_projection_engine": {
        "warp_ends_per_cm": pred_warp_proj,
        "weft_picks_per_cm": pred_weft_proj,
        "mean_warp_spacing_um": float(round(mean_warp_spacing_um, 2)),
        "mean_weft_spacing_um": float(round(mean_weft_spacing_um, 2))
    },
    "consensus_output": {
        "validated_warp_ends_per_cm": final_warp_count,
        "validated_weft_picks_per_cm": final_weft_count,
        "status_consistent": bool(delta_warp <= 1 and delta_weft <= 1)
    }
}

json_file = os.path.join(WORKING_DIR, "output.json")
txt_file = os.path.join(WORKING_DIR, "output.txt")

with open(json_file, "w") as f:
    json.dump(results_payload, f, indent=2)

with open(txt_file, "w") as f:
    f.write(f"=== FABRIC THREAD DENSITY ANALYSIS REPORT ===\n")
    f.write(f"Scale: {pixels_per_cm} px/cm | Resolution: {10000.0 / pixels_per_cm} um/px\n")
    f.write(f"True Warp: {true_warp_count} | Pred Warp (FFT): {pred_warp_fft} | Pred Warp (Proj): {pred_warp_proj}\n")
    f.write(f"True Weft: {true_weft_count} | Pred Weft (FFT): {pred_weft_fft} | Pred Weft (Proj): {pred_weft_proj}\n")
    f.write(f"Validated Result: {final_warp_count} Ends/cm (Warp) | {final_weft_count} Picks/cm (Weft)\n")
    f.write(f"Consistency Check: {results_payload['consensus_output']['status_consistent']}\n")

print(json.dumps(results_payload, indent=2))
print(f"Results persisted to: {json_file} and {txt_file}")

{
  "calibration": {
    "pixels_per_cm": 400,
    "pixel_size_um": 25.0,
    "roi_dimensions": [
      400,
      400
    ]
  },
  "ground_truth": {
    "warp_ends_per_cm": 32,
    "weft_picks_per_cm": 28
  },
  "spectral_fft_engine": {
    "warp_ends_per_cm": 32,
    "weft_picks_per_cm": 28,
    "error_warp": 0,
    "error_weft": 0
  },
  "spatial_projection_engine": {
    "warp_ends_per_cm": 31,
    "weft_picks_per_cm": 27,
    "mean_warp_spacing_um": 313.33,
    "mean_weft_spacing_um": 357.69
  },
  "consensus_output": {
    "validated_warp_ends_per_cm": 32,
    "validated_weft_picks_per_cm": 28,
    "status_consistent": true
  }
}
Results persisted to: results/output.json and results/output.txt
